# Mandatory Assignment 2
## Natural Language Processing and Text Analytics (KAN-CDSCO1002U)

*Group: *

*Student IDs: 185912, 161989, 160714 & 160363*

*Dataset: CyberBullying Comments Dataset*

---
# Part 1: Extended Abstract

## Lexicon-Based Sentiment Analysis: Foundations, Applications, and Limitations

### Introduction

Sentiment analysis refers to the computational task of identifying and extracting subjective information from text, typically classifying expressed opinions as positive, negative, or neutral (Liu, 2012). Among the established approaches, lexicon-based sentiment analysis stands out for its transparency, interpretability, and independence from labeled training data. Rather than learning sentiment patterns from annotated corpora, lexicon-based methods rely on predefined dictionaries that map words or phrases to sentiment scores. This extended abstract outlines the foundational principles of lexicon-based sentiment analysis, surveys its application across domains, and discusses its strengths and limitations relative to machine learning approaches.

### Foundations and Mechanism

A sentiment lexicon is a structured resource that associates words with polarity values. Well-known examples include SentiWordNet (Baccianella et al., 2010), the AFINN lexicon (Nielsen, 2011), and VADER (Hutto & Gilbert, 2014). The core mechanism is straightforward: given an input text, each token is matched against the lexicon, and the sentiment of the document is computed as an aggregation of individual word scores.

VADER (Valence Aware Dictionary and sEntiment Reasoner) extends this basic approach by incorporating grammatical and syntactical heuristics. It accounts for intensifiers ("very good"), negations ("not bad"), punctuation emphasis, and capitalisation, making it particularly suited for informal text such as social media posts (Hutto & Gilbert, 2014). AFINN, by contrast, assigns integer scores ranging from -5 to +5 to approximately 2,477 English words, offering a simpler but effective scoring system (Nielsen, 2011).

The general workflow involves (1) text preprocessing, including tokenisation and optional lowercasing, (2) token-level lookup against the lexicon, and (3) score aggregation at the document level, often through summation or averaging.

### Applications Across Domains

Lexicon-based sentiment analysis has been applied extensively across domains where labeled data is scarce or where interpretability is paramount.

In **financial text mining**, Loughran and McDonald (2011) demonstrated that general-purpose sentiment lexicons perform poorly on financial texts because words like "liability" or "risk" carry negative connotations in everyday language but are neutral in financial contexts. Their domain-specific lexicon has since become a standard resource for analysing earnings calls, 10-K filings, and financial news.

In **social media monitoring**, VADER has been widely adopted for analysing Twitter data due to its sensitivity to informal language features such as slang, emoticons, and abbreviation patterns (Hutto & Gilbert, 2014). Public health researchers have used lexicon-based tools to track sentiment toward vaccination campaigns and disease outbreaks in real time (Salathé & Khandelwal, 2011).

In **product and service reviews**, lexicon-based approaches serve as baselines for opinion mining systems. Taboada et al. (2011) proposed the Semantic Orientation Calculator (SO-CAL), which applies intensification and negation rules on top of a manually curated lexicon to achieve competitive performance on review datasets.

### Strengths and Limitations

The primary strength of lexicon-based methods is their **transparency**. Each sentiment score can be traced back to specific words and rules, which supports human interpretability and domain validation. Additionally, these methods require **no labeled training data**, making them immediately applicable to new domains or languages provided a suitable lexicon exists.

However, lexicon-based methods face notable limitations. They struggle with **context-dependent meaning**, such as sarcasm and irony, where surface-level word polarity diverges from intended sentiment. They are also sensitive to **domain mismatch**, as the same word may carry different sentiment in different fields. Coverage gaps remain a concern, particularly for specialised terminology, neologisms, and code-switched text.

Compared to supervised machine learning approaches such as Naive Bayes or Logistic Regression, lexicon-based methods tend to achieve lower accuracy on well-defined benchmark tasks. However, they remain competitive in low-resource settings and provide a useful baseline against which data-driven models can be evaluated.

### Conclusion

Lexicon-based sentiment analysis offers an accessible, interpretable, and data-efficient approach to opinion mining. Its reliance on curated dictionaries and transparent aggregation rules makes it well-suited for exploratory analysis and domains where labeled data is unavailable. While supervised and deep learning methods have surpassed lexicon-based approaches on many benchmarks, the simplicity and explainability of the lexicon paradigm ensure its continued relevance in both research and applied NLP.

### References

Baccianella, S., Esuli, A., & Sebastiani, F. (2010). SentiWordNet 3.0: An enhanced lexical resource for sentiment analysis and opinion mining. *Proceedings of LREC'10*, 2200-2204.

Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of the International AAAI Conference on Web and Social Media, 8*(1), 216-225.

Liu, B. (2012). Sentiment analysis and opinion mining. *Synthesis Lectures on Human Language Technologies, 5*(1), 1-167.

Loughran, T., & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. *The Journal of Finance, 66*(1), 35-65.

Nielsen, F. A. (2011). A new ANEW: Evaluation of a word list for sentiment analysis in microblogs. *Proceedings of the ESWC2011 Workshop on Making Sense of Microposts*, 93-98.

Salathé, M., & Khandelwal, S. (2011). Assessing vaccination sentiments with online social media. *PLoS Computational Biology, 7*(10), e1002199.

Taboada, M., Brooke, J., Tofiloski, M., Voll, K., & Stede, M. (2011). Lexicon-based methods for sentiment analysis. *Computational Linguistics, 37*(2), 267-307.

---
# Part 2: Cyberbullying Classification

Binary classification of social-media comments as **cyberbullying (1)** or **not cyberbullying (0)** using:
1. Multinomial Naive Bayes
2. Logistic Regression

**Pipeline:**
```
Load data -> Explore -> Preprocess -> Split (80/20) -> TF-IDF vectorise
         -> Naive Bayes -> Evaluate
         -> Logistic Regression -> Evaluate
         -> Compare
```

In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score,
                              precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

warnings.filterwarnings('ignore')

## 1. Data Preparation

### 1.1 Load and Explore the Dataset

The dataset contains two columns:

| Column | Type | Description |
|--------|------|-------------|
| `Text` | str | Raw social-media comment |
| `CB_Label` | int | 1 = cyberbullying, 0 = not cyberbullying |

In [ ]:
# Load the dataset -- place the CSV in the same directory as this notebook
df = pd.read_csv('CyberBullying_Comments_Dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumn names: {df.columns.tolist()}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Check class distribution
print('Class distribution:')
print(df['CB_Label'].value_counts())
print(f'\nClass proportions:')
print(df['CB_Label'].value_counts(normalize=True).round(4))

# Visualise class balance
# A perfectly balanced 50/50 split means accuracy is a reliable metric
# and no class-weight adjustments or oversampling (e.g. SMOTE) are needed
fig, ax = plt.subplots(figsize=(6, 4))
df['CB_Label'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Class Distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_xticklabels(['Not Cyberbullying (0)', 'Cyberbullying (1)'], rotation=0)
plt.tight_layout()
plt.show()

print('The dataset is perfectly balanced (50/50). No resampling required.')

### 1.2 Text Preprocessing

Raw social-media text is noisy. The following steps are applied in order:

| Step | Operation | Reason |
|------|-----------|--------|
| 1 | Lowercase | Normalise `"HATE"` and `"hate"` to the same token |
| 2 | Remove URLs | Hyperlinks carry no semantic value |
| 3 | Remove @mentions and #hashtags | Twitter-specific noise that references users or topics rather than expressing opinion |
| 4 | Keep letters only | Strip punctuation, digits, and residual symbols |
| 5 | Collapse whitespace | Tidy up extra spaces left after stripping |
| 6 | Remove stop words | Common words such as `"the"` and `"is"` add no discriminative signal |
| 7 | Drop short tokens | Tokens of length 2 or fewer are typically noise |

In [ ]:
# Stop-word list: standard English corpus plus Twitter-specific tokens.
# Built as a frozenset for O(1) lookup during tokenisation.
STOP_WORDS = frozenset([
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y','ain','aren','couldn','didn','doesn','hadn','hasn',
    'haven','isn','ma','mightn','mustn','needn','shan','shouldn','wasn',
    'weren','won','wouldn',
    # Twitter / social-media specific
    'rt', 'via', 'amp',
])


def preprocess(text: str) -> str:
    """
    Clean and tokenise a single comment string.

    Parameters
    ----------
    text : str
        Raw social-media comment.

    Returns
    -------
    str
        Space-joined list of clean tokens, ready for TF-IDF vectorisation.
    """
    # Step 1: lowercase the entire string
    text = str(text).lower()

    # Step 2: remove URLs (http://..., https://..., www....)
    text = re.sub(r'http\S+|www\S+', '', text)

    # Step 3: remove @mentions and #hashtags
    # These reference usernames and topics, not content, so they are noise
    text = re.sub(r'@\w+|#\w+', '', text)

    # Step 4: keep only ASCII letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 5: collapse multiple consecutive spaces to one
    text = re.sub(r'\s+', ' ', text).strip()

    # Steps 6 and 7: split, then drop stop words and very short tokens
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]

    # Rejoin into a single string (required by TfidfVectorizer)
    return ' '.join(tokens)

In [ ]:
# Apply preprocessing to every comment in the dataset
df['clean_text'] = df['Text'].apply(preprocess)

# Demonstrate the effect on a sample comment
idx = 10
print('Original :', df['Text'].iloc[idx])
print('Cleaned  :', df['clean_text'].iloc[idx])

# Some comments become empty strings after cleaning (e.g. a tweet that was
# purely a URL or a @mention). Drop these rows -- they carry no text.
empty_mask = df['clean_text'].str.strip() == ''
print(f'\nRows emptied by preprocessing: {empty_mask.sum()} (dropped)')
df = df[~empty_mask].reset_index(drop=True)
print(f'Usable rows after filtering: {len(df)}')

### 1.3 Train / Test Split (80% / 20%)

> **Important:** the data is split *before* fitting the vectoriser. If TF-IDF were fitted on the full dataset first, the IDF weights would be computed using test-set word frequencies, giving the model implicit knowledge of the test distribution and producing optimistically inflated metrics. Splitting first and fitting only on the training data prevents this.

`stratify=y` preserves the 50/50 class ratio in both partitions. `random_state=42` ensures full reproducibility.

In [ ]:
X = df['clean_text']   # features: cleaned text strings
y = df['CB_Label']     # target: 0 or 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # 20% held out for testing
    random_state=42,    # fixed seed for reproducibility
    stratify=y          # preserve the 50/50 class ratio in both splits
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'\nTrain label distribution:\n{y_train.value_counts()}')
print(f'\nTest label distribution:\n{y_test.value_counts()}')

### 1.4 TF-IDF Vectorisation

TF-IDF (Term Frequency-Inverse Document Frequency) assigns a weight to each n-gram that reflects how often it appears in a specific document (TF) relative to how rare it is across all documents (IDF). Words that are common everywhere receive near-zero weights; distinctive words receive high weights.

| Hyperparameter | Value | Reason |
|---|---|---|
| `max_features` | 10,000 | Cap vocabulary at the 10,000 most informative n-grams to manage memory and reduce noise |
| `ngram_range` | (1, 2) | Include bigrams alongside unigrams; phrases like *"shut up"* or *"kill yourself"* carry bullying intent that individual words alone do not |
| `min_df` | 2 | Ignore n-grams appearing in fewer than 2 documents (likely typos or one-off noise) |
| `sublinear_tf` | True | Replace raw TF with 1+log(TF), reducing the dominance of very high-frequency terms |

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10_000,  # limit vocabulary size
    ngram_range=(1, 2),   # unigrams AND bigrams
    min_df=2,             # ignore very rare n-grams
    sublinear_tf=True     # log-scaled term frequency
)

# fit_transform on train: learns vocabulary and IDF weights from training data ONLY
X_train_tfidf = tfidf.fit_transform(X_train)

# transform on test: applies the same vocabulary/IDF without refitting (no leakage)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF feature matrix (train): {X_train_tfidf.shape}')
print(f'TF-IDF feature matrix (test) : {X_test_tfidf.shape}')

---
## 2. Naive Bayes Classification

We use **Multinomial Naive Bayes (MNB)** because our features are TF-IDF scores: non-negative, count-like values that fit the multinomial distribution assumption.

| Variant | Best suited for | Our choice |
|---------|-----------------|------------|
| `MultinomialNB` | TF-IDF / word-count features | Yes |
| `BernoulliNB` | Binary word-presence features | No |
| `GaussianNB` | Continuous numerical features | No |

`alpha=0.1` applies Laplace smoothing: a small pseudo-count added to every feature so that n-grams unseen during training do not produce a zero probability at inference time. The default `alpha=1.0` over-smooths on larger datasets; 0.1 is a lighter touch that works better here.

In [ ]:
# Train MultinomialNB
# alpha=0.1 applies light Laplace smoothing (default 1.0 over-smooths on this dataset)
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

# Predict class labels for the test set
nb_preds = nb_model.predict(X_test_tfidf)

# Compute metrics
nb_accuracy  = accuracy_score(y_test, nb_preds)
nb_precision = precision_score(y_test, nb_preds)  # positive class = 1 (cyberbullying)
nb_recall    = recall_score(y_test, nb_preds)
nb_f1        = f1_score(y_test, nb_preds)

print('Naive Bayes -- Test-Set Performance')
print(f'  Accuracy  : {nb_accuracy:.4f}')
print(f'  Precision : {nb_precision:.4f}  (of all predicted CB, how many are truly CB?)')
print(f'  Recall    : {nb_recall:.4f}  (of all true CB, how many did we catch?)')
print(f'  F1-Score  : {nb_f1:.4f}  (harmonic mean of precision and recall)')
print('\nDetailed classification report:')
print(classification_report(y_test, nb_preds,
                             target_names=['Not Cyberbullying', 'Cyberbullying']))

In [ ]:
# Confusion matrix heatmap for Naive Bayes
fig, ax = plt.subplots(figsize=(6, 5))
cm_nb = confusion_matrix(y_test, nb_preds)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
ax.set_title('Naive Bayes -- Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()
print(f'TN={cm_nb[0,0]}  FP={cm_nb[0,1]}  FN={cm_nb[1,0]}  TP={cm_nb[1,1]}')

---
## 3. Logistic Regression Classification

Logistic Regression (LR) is a discriminative linear classifier. Unlike Naive Bayes, it does not assume feature independence: all feature weights are jointly optimised by maximising the log-likelihood of the training labels. This makes LR generally more robust when features are correlated, as is common in n-gram spaces.

| Hyperparameter | Value | Reason |
|---|---|---|
| `C` | 1.0 | Inverse regularisation strength; default provides a moderate L2 penalty |
| `solver` | `lbfgs` | Efficient for medium-sized dense problems; supports L2 regularisation natively |
| `max_iter` | 1000 | The default of 100 is often insufficient for high-dimensional TF-IDF matrices |
| `random_state` | 42 | Reproducibility |

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(
    C=1.0,           # inverse regularisation strength
    solver='lbfgs',  # Limited-memory BFGS optimiser
    max_iter=1000,   # allow enough iterations to converge on TF-IDF features
    random_state=42
)
lr_model.fit(X_train_tfidf, y_train)

# Predict class labels for the test set
lr_preds = lr_model.predict(X_test_tfidf)

# Compute metrics
lr_accuracy  = accuracy_score(y_test, lr_preds)
lr_precision = precision_score(y_test, lr_preds)
lr_recall    = recall_score(y_test, lr_preds)
lr_f1        = f1_score(y_test, lr_preds)

print('Logistic Regression -- Test-Set Performance')
print(f'  Accuracy  : {lr_accuracy:.4f}')
print(f'  Precision : {lr_precision:.4f}  (of all predicted CB, how many are truly CB?)')
print(f'  Recall    : {lr_recall:.4f}  (of all true CB, how many did we catch?)')
print(f'  F1-Score  : {lr_f1:.4f}  (harmonic mean of precision and recall)')
print('\nDetailed classification report:')
print(classification_report(y_test, lr_preds,
                             target_names=['Not Cyberbullying', 'Cyberbullying']))

In [ ]:
# Confusion matrix heatmap for Logistic Regression
fig, ax = plt.subplots(figsize=(6, 5))
cm_lr = confusion_matrix(y_test, lr_preds)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
ax.set_title('Logistic Regression -- Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()
print(f'TN={cm_lr[0,0]}  FP={cm_lr[0,1]}  FN={cm_lr[1,0]}  TP={cm_lr[1,1]}')

---
## 4. Model Comparison Summary

In [ ]:
# Side-by-side comparison table with winner column
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Naive Bayes': [nb_accuracy, nb_precision, nb_recall, nb_f1],
    'Logistic Regression': [lr_accuracy, lr_precision, lr_recall, lr_f1]
})
results[['Naive Bayes', 'Logistic Regression']] = (
    results[['Naive Bayes', 'Logistic Regression']].round(4)
)
results['Winner'] = results.apply(
    lambda row: 'NB' if row['Naive Bayes'] > row['Logistic Regression']
               else ('LR' if row['Logistic Regression'] > row['Naive Bayes'] else 'Tie'),
    axis=1
)
print('=== Model Comparison ===\n')
print(results.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results['Metric']))
width = 0.35
ax.bar(x - width/2, results['Naive Bayes'], width,
       label='Naive Bayes', color='steelblue')
ax.bar(x + width/2, results['Logistic Regression'], width,
       label='Logistic Regression', color='coral')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results['Metric'])
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 4. Short Report: Model Comparison

### Performance Overview

Both Naive Bayes (MultinomialNB) and Logistic Regression were trained on TF-IDF features extracted from the CyberBullying Comments Dataset (11,100 samples, perfectly balanced at 50/50). The table and chart above present accuracy, precision, recall, and F1-score on the held-out 20% test set (2,211 samples).

Logistic Regression outperforms Naive Bayes on three out of four metrics: accuracy (0.7096 vs 0.6888), precision (0.7283 vs 0.6951), and F1-score (0.6989 vs 0.6856). Naive Bayes achieves marginally higher recall on the cyberbullying class (0.6763 vs 0.6718), but the difference is negligible (0.45 percentage points). Overall, Logistic Regression is the stronger model on this dataset.

### Advantages and Disadvantages

**Naive Bayes**

| | |
|---|---|
| **Advantage 1** | Very fast to train and predict, making it practical for large-scale or real-time content moderation. Parameters are estimated analytically in a single pass; no iterative optimisation is required. |
| **Advantage 2** | Achieves marginally higher recall on the cyberbullying class (0.6763 vs 0.6718). In a harm-reduction context, failing to flag a genuine bullying comment is more costly than a false alarm, so recall is arguably the most safety-critical metric. |
| **Disadvantage 1** | The conditional independence assumption ignores word co-occurrence and phrase-level patterns, limiting its ability to capture nuanced cyberbullying expressions such as sarcasm or coded language. |
| **Disadvantage 2** | Produces poorly calibrated probability estimates, which makes threshold tuning for content moderation applications less reliable. |

**Logistic Regression**

| | |
|---|---|
| **Advantage 1** | Achieves higher accuracy (0.7096 vs 0.6888) and precision on the cyberbullying class (0.7283 vs 0.6951), meaning fewer innocent comments are falsely flagged. This is preferable in settings where false accusations carry a high social cost. |
| **Advantage 2** | Produces well-calibrated probability estimates, useful for adjusting classification thresholds in sensitive applications. Feature coefficients are also directly interpretable: the largest positive weights identify which words and bigrams most strongly indicate cyberbullying. |
| **Disadvantage 1** | Slower to train than Naive Bayes due to iterative L-BFGS optimisation over the full feature space. |
| **Disadvantage 2** | Marginally lower recall (0.6718 vs 0.6763), meaning it misses slightly more genuine cyberbullying comments. In a safety-critical context this is a meaningful trade-off to consider. |

### Additional Findings

The dataset is perfectly balanced (50/50 split), eliminating class imbalance as a confounding factor and making accuracy a reliable metric alongside F1-score. Both models achieve accuracy in the range of 69-71%, indicating that cyberbullying detection from text alone is a moderately difficult task. Many comments likely contain ambiguous language where context, tone, or sarcasm determines whether the text is bullying.

The inclusion of bigrams (`ngram_range=(1,2)`) adds meaningful signal: phrases such as *"shut up"*, *"go die"*, or *"kill yourself"* carry clear bullying intent that individual unigrams alone do not capture. Improving beyond this baseline would likely require richer representations such as word embeddings or transformer-based models (e.g. BERT) that capture sequential and contextual meaning.

### References

Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of the International AAAI Conference on Web and Social Media, 8*(1), 216-225.

Jurafsky, D., & Martin, J. H. (2014). *Speech and language processing* (2nd ed.). Pearson.

Manning, C. D., & Schutze, H. (2003). *Foundations of statistical natural language processing*. MIT Press.

Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830.

Sooraj Tomar (2023). Cyberbullying Tweets Dataset. Kaggle. https://www.kaggle.com/datasets/soorajtomar/cyberbullying-tweets